<a href="https://colab.research.google.com/github/nikhilmooloo02-stack/CLARIX_AI_AGENT/blob/main/CLARIX_AI_AGENT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
!pip install anthropic gradio chromadb -q
print("Packages ready")

Packages ready


In [4]:
import anthropic
import chromadb
from google.colab import userdata

# CONNECTION
client = anthropic.Anthropic(api_key=userdata.get('ANTHROPIC_API_KEY'))

# PRODUCT KNOWLEDGE BASE
chroma_client = chromadb.Client()
knowledge_base = chroma_client.create_collection(name="shaneal_products")

products = """
PAPER:
- Butterfly A4 Paper 80gsm 500 sheets - R89.00
- Rotatrim A4 Paper 75gsm 500 sheets - R75.00
- A3 Paper 80gsm 500 sheets - R165.00
- Letterhead Paper A4 100 sheets - R45.00

PENS AND WRITING:
- Bic Ballpoint Pens Blue Box of 50 - R120.00
- Pilot G2 Gel Pens Black Pack of 12 - R95.00
- Staedtler Permanent Markers Pack of 10 - R85.00
- Highlighters Assorted Pack of 5 - R55.00

PPE:
- Surgical Face Masks Box of 50 - R95.00
- Nitrile Gloves Box of 100 - R180.00
- Safety Goggles - R45.00
- Reflective Safety Vest - R120.00

OFFICE EQUIPMENT:
- Bantex A4 Lever Arch File - R45.00
- Stapler Heavy Duty - R135.00
- Calculator Scientific - R250.00
- Whiteboard A1 - R850.00
- Shredder 10 Sheet - R1200.00

HOUSEHOLD CONSUMABLES:
- Refuse Bags Black Roll of 20 - R35.00
- Hand Sanitiser 500ml - R65.00
- Multipurpose Cleaning Spray 750ml - R45.00
- Toilet Paper 9 Roll Pack - R55.00
"""

knowledge_base.add(
documents=[products],
ids=["shaneal_product_list"]
)

# SYSTEM PROMPT
SYSTEM_PROMPT = """You are CLARIX, the professional AI assistant for
ShaNeal Distributors — a stationery, PPE, office equipment and household
consumables distributor based in Pretoria, South Africa.

You serve retail customers and bulk business and school clients.

PRODUCTS AND PRICING:
PAPER: Butterfly A4 Paper 80gsm 500 sheets R89.00 | Rotatrim A4 Paper
75gsm 500 sheets R75.00 | A3 Paper 80gsm 500 sheets R165.00 |
Letterhead Paper A4 100 sheets R45.00
PENS: Bic Ballpoint Pens Blue Box of 50 R120.00 | Pilot G2 Gel Pens
Black Pack of 12 R95.00 | Staedtler Permanent Markers Pack of 10 R85.00
| Highlighters Assorted Pack of 5 R55.00
PPE: Surgical Face Masks Box of 50 R95.00 | Nitrile Gloves Box of 100
R180.00 | Safety Goggles R45.00 | Reflective Safety Vest R120.00
OFFICE EQUIPMENT: Bantex A4 Lever Arch File R45.00 | Stapler Heavy Duty
R135.00 | Calculator Scientific R250.00 | Whiteboard A1 R850.00 |
Shredder 10 Sheet R1200.00
HOUSEHOLD CONSUMABLES: Refuse Bags Black Roll of 20 R35.00 | Hand
Sanitiser 500ml R65.00 | Multipurpose Cleaning Spray 750ml R45.00 |
Toilet Paper 9 Roll Pack R55.00

CONTACT INFORMATION:
- Phone: 070 070 0770
- Email: ShaNeal@lantic.co.za
- Address: 332 Paul Kruger Street, Corner Van Heerden Street,
Capital Park, Pretoria 0084
- Website: www.ShaNealonline.co.za
- Business Hours: Monday to Friday, 8am to 6pm

BEHAVIOUR:
- Recommend specific products with prices
- Cross-sell related products naturally
- For bulk orders ask for organisation name and delivery address
- For complaints apologise sincerely and offer a clear solution
- For orders over R10,000 escalate to a human consultant
- When customers ask for contact details share all of the above clearly
- When customers want to speak to a person provide phone and email
- Always be warm, professional and solution-focused"""

print("CLARIX is ready")

/root/.cache/chroma/onnx_models/all-MiniLM-L6-v2/onnx.tar.gz: 100%|██████████| 79.3M/79.3M [00:02<00:00, 29.2MiB/s]


CLARIX is ready


In [6]:
import gradio as gr

def chat(message, history):
    results = knowledge_base.query(query_texts=[message], n_results=1)
    product_context = ""
    if results['documents'][0]:
        product_context = results['documents'][0][0]

    enhanced_message = f"""Customer query: {message}
ShaNeal product information: {product_context}"""

    conversation = []
    for item in history:
        conversation.append({"role": "user", "content": item[0]})
        conversation.append({"role": "assistant", "content": item[1]})
    conversation.append({"role": "user", "content": enhanced_message})

    response = client.messages.create(
        model="claude-sonnet-4-5",
        max_tokens=800,
        system=SYSTEM_PROMPT,
        messages=conversation
    )
    return response.content[0].text

gr.ChatInterface(
    fn=chat,
    title="CLARIX — ShaNeal Distributors AI Assistant",
    description="Ask about products, pricing, bulk orders, or get support.",
    examples=[
        "How can I contact ShaNeal Distributors?",
        "What pens do you have and how much do they cost?",
        "I need PPE for 20 staff members.",
        "My order arrived damaged, I need a refund.",
    ],
    theme=gr.themes.Soft()
).launch(share=True)

/usr/local/lib/python3.12/dist-packages/gradio/chat_interface.py:347: UserWarning: The 'tuples' format for chatbot messages is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style 'role' and 'content' keys.
  self.chatbot = Chatbot(


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://d0cffcfda06a4c7089.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
